In [20]:
import random
import os
from genedesign.rbs_chooser import RBSChooser
from genedesign.models.transcript import Transcript
from genedesign.checkers.codon_checker import CodonChecker
from genedesign.checkers.forbidden_sequence_checker import ForbiddenSequenceChecker
# hairpin_checker is a function; PromoterChecker is a class
from genedesign.checkers.hairpin_checker import hairpin_checker
from genedesign.checkers.internal_promoter_checker import PromoterChecker


class TranscriptDesigner:
    """
    Reverse translates a protein sequence into a DNA sequence and selects an RBS.
    Validated against multiple biological constraints with robust error handling.
    """

    MAX_ATTEMPTS = 1000

    def __init__(self):
        self.codon_table = {}
        self.rbsChooser = None
        self.codonChecker = None
        self.forbiddenChecker = None
        self.promoterChecker = None

    def initiate(self) -> None:
        """
        Initializes the codon usage table and class-based checkers.
        Note: hairpin_checker is a function and does not need initiation.
        """
        self.rbsChooser = RBSChooser()
        self.rbsChooser.initiate()

        self.codonChecker = CodonChecker()
        self.codonChecker.initiate()

        self.forbiddenChecker = ForbiddenSequenceChecker()
        self.forbiddenChecker.initiate()

        self.promoterChecker = PromoterChecker()
        self.promoterChecker.initiate()

        try:
            self.codon_table = self._load_codon_usage()
        except Exception:
            self.codon_table = self._default_codon_table()

    def run(self, peptide: str, ignores: set) -> Transcript:
        """
        Translates peptide to DNA, ensuring it passes all checker constraints.
        """
        for attempt in range(1, self.MAX_ATTEMPTS + 1):
            codons = self._sample_codons(peptide)
            cds = ''.join(codons)

            # Validate the sequence
            failures = self._check_sequence(cds)
            if not failures:
                # Safe call to RBS Chooser
                selected_rbs = self.rbsChooser.run(cds, ignores)
                return Transcript(selected_rbs, peptide, codons)

        raise RuntimeError(f"Failed to find valid sequence after {self.MAX_ATTEMPTS} attempts.")

    def _sample_codons(self, peptide: str) -> list[str]:
        """Weighted random sampling of codons based on E. coli frequency."""
        codons = []
        for aa in peptide:
            options = self.codon_table.get(aa)
            if not options:
                raise ValueError(f"Unknown amino acid: '{aa}'")
            
            c_list = [opt[0] for opt in options]
            w_list = [opt[1] for opt in options]
            codons.append(random.choices(c_list, weights=w_list, k=1)[0])

        # Append weighted stop codon
        stop_options = self.codon_table.get('*', [('TAA', 0.61), ('TAG', 0.09), ('TGA', 0.30)])
        s_list = [opt[0] for opt in stop_options]
        sw_list = [opt[1] for opt in stop_options]
        codons.append(random.choices(s_list, weights=sw_list, k=1)[0])
        return codons

    def _check_sequence(self, cds: str) -> list[str]:
        """
        Runs all checkers with safe unpacking to prevent 'too many values to unpack' errors.
        """
        failures = []
        
        # 1. Class-based checkers (Return 2 or more values)
        checkers = [
            (self.forbiddenChecker, "Forbidden"), 
            (self.promoterChecker, "Promoter"), 
            (self.codonChecker, "Codon")
        ]

        for checker, label in checkers:
            # result might be (True, "Reason") or (True, "Reason", metadata...)
            result = checker.run(cds)
            
            # Safe unpack: status is always first, reason is second
            ok = result[0]
            reason = result[1] if len(result) > 1 else "No reason provided"
            
            if not ok:
                failures.append(f"{label}: {reason}")
        
        # 2. Function-based checker (hairpin_checker)
        # Returns (bool, str or None)
        ok, reason = hairpin_checker(cds)
        if not ok:
            failures.append(f"Hairpin: {reason}")
        
        return failures

    def _load_codon_usage(self) -> dict:
        """Parses the local codon_usage.txt file if available."""
        import os
        data_path = os.path.join(os.path.dirname(__file__), 'data', 'codon_usage.txt')
        aa3to1 = {
            'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
            'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
            'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
            'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V',
            'Ter': '*',
        }
        table = {}
        if os.path.exists(data_path):
            with open(data_path) as fh:
                for line in fh:
                    parts = line.strip().split()
                    if len(parts) >= 3 and parts[1] in aa3to1:
                        aa1 = aa3to1[parts[1]]
                        table.setdefault(aa1, []).append((parts[0], float(parts[2])))
        return table if table else self._default_codon_table()

    @staticmethod
    def _default_codon_table() -> dict:
        """Fallback E. coli codon table."""
        return {
            'A': [('GCT', 0.18), ('GCC', 0.26), ('GCA', 0.23), ('GCG', 0.33)],
            'R': [('CGT', 0.36), ('CGC', 0.40), ('CGA', 0.07), ('CGG', 0.11), ('AGA', 0.04), ('AGG', 0.02)],
            'N': [('AAT', 0.45), ('AAC', 0.55)], 'D': [('GAT', 0.63), ('GAC', 0.37)],
            'C': [('TGT', 0.45), ('TGC', 0.55)], 'Q': [('CAA', 0.34), ('CAG', 0.66)],
            'E': [('GAA', 0.68), ('GAG', 0.32)], 'G': [('GGT', 0.35), ('GGC', 0.40), ('GGA', 0.11), ('GGG', 0.14)],
            'H': [('CAT', 0.57), ('CAC', 0.43)], 'I': [('ATT', 0.49), ('ATC', 0.39), ('ATA', 0.11)],
            'L': [('TTA', 0.14), ('TTG', 0.13), ('CTT', 0.12), ('CTC', 0.10), ('CTA', 0.04), ('CTG', 0.47)],
            'K': [('AAA', 0.74), ('AAG', 0.26)], 'M': [('ATG', 1.00)], 'F': [('TTT', 0.58), ('TTC', 0.42)],
            'P': [('CCT', 0.18), ('CCC', 0.13), ('CCA', 0.20), ('CCG', 0.49)],
            'S': [('TCT', 0.17), ('TCC', 0.15), ('TCA', 0.14), ('TCG', 0.14), ('AGT', 0.16), ('AGC', 0.25)],
            'T': [('ACT', 0.19), ('ACC', 0.40), ('ACA', 0.17), ('ACG', 0.25)],
            'W': [('TGG', 1.00)], 'Y': [('TAT', 0.59), ('TAC', 0.41)],
            'V': [('GTT', 0.28), ('GTC', 0.20), ('GTA', 0.17), ('GTG', 0.35)],
            '*': [('TAA', 0.61), ('TAG', 0.09), ('TGA', 0.30)],
        }

# ---------------------------------------------------------------------------
# Simple Test Logic
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    designer = TranscriptDesigner()
    designer.initiate()

    test_peptide = "MYPFIRTARMTV"
    
    print(f"--- Testing TranscriptDesigner ---")
    print(f"Input Peptide: {test_peptide}")

    try:
        transcript = designer.run(test_peptide, ignores=set())
        dna_sequence = "".join(transcript.codons)
        
        print(f"Success!")
        print(f"Selected RBS: {transcript.rbs}")
        print(f"Generated DNA: {dna_sequence}")
        
        # Validation
        expected_len = (len(test_peptide) + 1) * 3
        assert len(dna_sequence) == expected_len, f"Length mismatch!"
        assert dna_sequence[-3:] in ["TAA", "TAG", "TGA"], "Missing stop codon!"
        
        print("Test passed: Sequence is valid.")

    except Exception as e:
        print(f"Test failed with error: {e}")

--- Testing TranscriptDesigner ---
Input Peptide: MYPFIRTARMTV
Test failed with error: Failed to find valid sequence after 1000 attempts.
